In [30]:
import polars as pl
import folium

In [31]:
#Read df to use
df = pl.read_parquet("../data/output/v2/2.clean_df_v2.parquet")

In [32]:
#-------------------------------
#le nombre total de trajets en 2025 ;
#--------------------------------

print(df.select(pl.count("Total duration (ms)")))

#After cleaning the data, there were 8'762,355 trips made in 2025


shape: (1, 1)
┌─────────────────────┐
│ Total duration (ms) │
│ ---                 │
│ u32                 │
╞═════════════════════╡
│ 8762355             │
└─────────────────────┘


In [33]:
#-------------------------------
# le nombre ou le ratio de stations actives ;
#-------------------------------

#number of active start stations 
print(df.select(pl.col("Start station number").unique_counts()))

#820 start stations where used in 2025

#number of active end stations
print(df.select(pl.col("End station number").unique_counts()))

#822 end stations where used in 2025

#Note, this could change if the 2 stations that only appear in the end station number column are removed in the cleaning step.(station ids 10626 and 22168)




shape: (820, 1)
┌──────────────────────┐
│ Start station number │
│ ---                  │
│ u32                  │
╞══════════════════════╡
│ 15056                │
│ 13531                │
│ 11088                │
│ 13191                │
│ 15764                │
│ …                    │
│ 237                  │
│ 3331                 │
│ 3125                 │
│ 841                  │
│ 37                   │
└──────────────────────┘
shape: (822, 1)
┌────────────────────┐
│ End station number │
│ ---                │
│ u32                │
╞════════════════════╡
│ 10167              │
│ 7337               │
│ 11464              │
│ 22345              │
│ 10075              │
│ …                  │
│ 4396               │
│ 3257               │
│ 3180               │
│ 831                │
│ 27                 │
└────────────────────┘


In [34]:
#Rate of active stations
start_ids = df["Start station number"].unique()
end_ids = df["End station number"].unique()

all_stations = pl.concat([start_ids, end_ids]).unique()

start_rate = start_ids.len() / all_stations.len()
end_rate = end_ids.len() / all_stations.len()

print(f"Start station active rate: {start_rate:.2%}")
print(f"End station active rate:   {end_rate:.2%}")

#almost 100% on both. which is normal actually since the empty stations where already filtered out



Start station active rate: 99.76%
End station active rate:   100.00%


In [35]:
#Most used start and end station
print(df.group_by("Start station number").len().sort("len", descending=True))

print(df.group_by("End station number").len().sort("len", descending=True))


shape: (820, 2)
┌──────────────────────┬───────┐
│ Start station number ┆ len   │
│ ---                  ┆ ---   │
│ i64                  ┆ u32   │
╞══════════════════════╪═══════╡
│ 1072                 ┆ 53880 │
│ 2696                 ┆ 46257 │
│ 1075                 ┆ 42737 │
│ 2587                 ┆ 39851 │
│ 1011                 ┆ 38519 │
│ …                    ┆ …     │
│ 200083444            ┆ 488   │
│ 11274444             ┆ 322   │
│ 1219444              ┆ 250   │
│ 200191444            ┆ 237   │
│ 2635                 ┆ 37    │
└──────────────────────┴───────┘
shape: (822, 2)
┌────────────────────┬───────┐
│ End station number ┆ len   │
│ ---                ┆ ---   │
│ i64                ┆ u32   │
╞════════════════════╪═══════╡
│ 1072               ┆ 50563 │
│ 2696               ┆ 44261 │
│ 960                ┆ 44106 │
│ 1161               ┆ 41383 │
│ 1067               ┆ 40483 │
│ …                  ┆ …     │
│ 11274444           ┆ 274   │
│ 1219444            ┆ 242   │
│ 20

In [36]:
# -------------------------------
# le nombre ou le ratio de stations utilisées comme stations de départ ;
#-------------------------------

#check previous point


#-------------------------------
# le nombre ou le ratio de stations utilisées comme stations d'arrivée ;
#-------------------------------

#check previous point




In [37]:
#-------------------------------
# la distribution du nombre de trajets par jour ;
#-------------------------------

#Change date format to date time
#df = df.with_columns(
#    pl.col("Start date").str.to_datetime("%Y-%m-%d %H:%M"),
#    pl.col("End date").str.to_datetime("%Y-%m-%d %H:%M")
#)
#print(df)


print(df.group_by("Operating date").len().sort("len", descending=True))

#there is a huge gap between the 2 lowest dates: 2025-08-05 with 171 trajets and 2025-01-01 with 5077 trajets (could it be that the missing values removed in the cleaning belonged mostly to 2025-08-05 ?)


shape: (365, 2)
┌────────────────┬───────┐
│ Operating date ┆ len   │
│ ---            ┆ ---   │
│ date           ┆ u32   │
╞════════════════╪═══════╡
│ 2025-09-09     ┆ 44732 │
│ 2025-09-11     ┆ 41080 │
│ 2025-09-08     ┆ 40110 │
│ 2025-09-10     ┆ 36695 │
│ 2025-07-10     ┆ 34028 │
│ …              ┆ …     │
│ 2025-12-24     ┆ 8216  │
│ 2025-12-26     ┆ 6171  │
│ 2025-01-05     ┆ 5739  │
│ 2025-01-01     ┆ 5077  │
│ 2025-08-05     ┆ 171   │
└────────────────┴───────┘


In [38]:
#-------------------------------
# la distribution du nombre de trajets par station ;
#-------------------------------

#Most used start and end station
print(df.group_by("Start station number").len().sort("len", descending=True))

print(df.group_by("End station number").len().sort("len", descending=True))


shape: (820, 2)
┌──────────────────────┬───────┐
│ Start station number ┆ len   │
│ ---                  ┆ ---   │
│ i64                  ┆ u32   │
╞══════════════════════╪═══════╡
│ 1072                 ┆ 53880 │
│ 2696                 ┆ 46257 │
│ 1075                 ┆ 42737 │
│ 2587                 ┆ 39851 │
│ 1011                 ┆ 38519 │
│ …                    ┆ …     │
│ 200083444            ┆ 488   │
│ 11274444             ┆ 322   │
│ 1219444              ┆ 250   │
│ 200191444            ┆ 237   │
│ 2635                 ┆ 37    │
└──────────────────────┴───────┘
shape: (822, 2)
┌────────────────────┬───────┐
│ End station number ┆ len   │
│ ---                ┆ ---   │
│ i64                ┆ u32   │
╞════════════════════╪═══════╡
│ 1072               ┆ 50563 │
│ 2696               ┆ 44261 │
│ 960                ┆ 44106 │
│ 1161               ┆ 41383 │
│ 1067               ┆ 40483 │
│ …                  ┆ …     │
│ 11274444           ┆ 274   │
│ 1219444            ┆ 242   │
│ 20

In [39]:
 
# -------------------------------
# la distribution des durées de trajets ;
#-------------------------------

print(df.group_by("Total duration (ms)").len().sort("len", descending=True))


shape: (2_418_292, 2)
┌─────────────────────┬─────┐
│ Total duration (ms) ┆ len │
│ ---                 ┆ --- │
│ i64                 ┆ u32 │
╞═════════════════════╪═════╡
│ 360953              ┆ 24  │
│ 340310              ┆ 23  │
│ 446246              ┆ 22  │
│ 471958              ┆ 22  │
│ 415027              ┆ 22  │
│ …                   ┆ …   │
│ 1931958             ┆ 1   │
│ 1832493             ┆ 1   │
│ 147417              ┆ 1   │
│ 7211738             ┆ 1   │
│ 3944017             ┆ 1   │
└─────────────────────┴─────┘


In [40]:
#-------------------------------
# la répartition des trajets selon le jour de la semaine ;
#-------------------------------

#Day of week being 1=monday and 7=sunday
print(df.group_by("Day of week").len().sort("len", descending=True))

#lower use of the bicycles is seen on the weekend (6=saturday and 7=sunday)


shape: (7, 2)
┌─────────────┬─────────┐
│ Day of week ┆ len     │
│ ---         ┆ ---     │
│ i8          ┆ u32     │
╞═════════════╪═════════╡
│ 3           ┆ 1417276 │
│ 4           ┆ 1411953 │
│ 2           ┆ 1407982 │
│ 1           ┆ 1257439 │
│ 5           ┆ 1242804 │
│ 6           ┆ 1073299 │
│ 7           ┆ 951602  │
└─────────────┴─────────┘


In [41]:
#-------------------------------
# la répartition des trajets selon l'heure de la journée.
#-------------------------------

print(df.group_by("Hour of day").len().sort("len", descending=True))

#Most used hours are before work and after work. Least used are the night/morning hours (horas de madrugada - heures de nuit)



shape: (24, 2)
┌─────────────┬────────┐
│ Hour of day ┆ len    │
│ ---         ┆ ---    │
│ i8          ┆ u32    │
╞═════════════╪════════╡
│ 17          ┆ 889343 │
│ 8           ┆ 865645 │
│ 18          ┆ 830044 │
│ 16          ┆ 593510 │
│ 9           ┆ 548452 │
│ …           ┆ …      │
│ 5           ┆ 45129  │
│ 1           ┆ 42485  │
│ 2           ┆ 25670  │
│ 4           ┆ 19820  │
│ 3           ┆ 18380  │
└─────────────┴────────┘


In [27]:
#######################
#bike map
#######################

# Chargement des stations
GPS_Point = pl.read_csv("../data/tfl_bike_stations.csv")

GPS_Point = GPS_Point[[
    "station_number",
    "station_name",
    "latitude",
    "longitude"
]]

# Centre approximatif de Londres
london_map = folium.Map(
    location=[51.5074, -0.1278],
    zoom_start=12
)

# Ajout des stations
for row in GPS_Point.drop_nulls(["latitude", "longitude"]).iter_rows(named=True):
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        popup=f'{row["station_name"]} - {row["station_number"]}',
        fill=True,
        fill_opacity=0.7
    ).add_to(london_map)



# Sauvegarde
london_map.save("stations_londres.html")

london_map

PROFILE STATION ANALYSIS


In [42]:
df

Start date,Start station number,End date,End station number,Total duration (ms),Operating date,Day of week,Hour of day,start_station_latitude,start_station_longitude,start_station_name,end_station_latitude,end_station_longitude,end_station_name
datetime[μs],i64,datetime[μs],i64,i64,date,i8,i8,f64,f64,str,f64,f64,str
2025-01-14 23:59:00,1043,2025-01-15 00:13:00,200149,870468,2025-01-14,2,23,51.517821,-0.096496,"""Museum of London, Barbican""",51.511542,-0.056667,"""Watney Street, Shadwell"""
2025-01-14 23:59:00,300015,2025-01-15 00:04:00,300229,307181,2025-01-14,2,23,51.472509,-0.122831,"""Binfield Road, Stockwell""",51.469202,-0.119022,"""Sidney Road, Stockwell"""
2025-01-14 23:59:00,1068,2025-01-15 00:10:00,1051,640755,2025-01-14,2,23,51.521113,-0.078869,"""Norton Folgate, Liverpool Stre…",51.534042,-0.086379,"""Shoreditch Park, Hoxton"""
2025-01-14 23:59:00,1159,2025-01-15 00:10:00,1007,642548,2025-01-14,2,23,51.522853,-0.099994,"""Berry Street, Clerkenwell""",51.51477,-0.122219,"""Drury Lane, Covent Garden"""
2025-01-14 23:58:00,200048,2025-01-15 00:14:00,1058,944185,2025-01-14,2,23,51.493978,-0.127554,"""Page Street, Westminster""",51.510212,0.004979,"""Orchard Place, Blackwall Tunne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-16 00:02:00,200141,2025-12-16 00:16:00,200237,837578,2025-12-16,2,0,51.509158,-0.224103,"""Westfield Ariel Way, White Cit…",51.472817,-0.199783,"""Parson's Green , Parson's Gree…"
2025-12-16 00:01:00,200011,2025-12-16 00:16:00,200250,908383,2025-12-16,2,0,51.519265,-0.021345,"""Furze Green, Bow""",51.520893,-0.051394,"""Cleveland Way, Stepney"""
2025-12-16 00:00:00,300096,2025-12-16 00:07:00,962,432164,2025-12-16,2,0,51.511891,-0.107349,"""Tallis Street, Temple""",51.505569,-0.111606,"""Stamford Street, South Bank"""


In [ ]:
df = df.rename({
    "Start date": "start_date",
    "Start station number": "start_station",
    "End date": "end_date",
    "End station number": "end_station",
    "Total duration (ms)": "duration_ms",
    "Operating date": "operating_date",
    "Day of week": "day_of_week",
    "Hour of day": "hour"
})



In [44]:
df

start_date,start_station,end_date,end_station,duration_ms,operating_date,day_of_week,hour,start_station_latitude,start_station_longitude,start_station_name,end_station_latitude,end_station_longitude,end_station_name
datetime[μs],i64,datetime[μs],i64,i64,date,i8,i8,f64,f64,str,f64,f64,str
2025-01-14 23:59:00,1043,2025-01-15 00:13:00,200149,870468,2025-01-14,2,23,51.517821,-0.096496,"""Museum of London, Barbican""",51.511542,-0.056667,"""Watney Street, Shadwell"""
2025-01-14 23:59:00,300015,2025-01-15 00:04:00,300229,307181,2025-01-14,2,23,51.472509,-0.122831,"""Binfield Road, Stockwell""",51.469202,-0.119022,"""Sidney Road, Stockwell"""
2025-01-14 23:59:00,1068,2025-01-15 00:10:00,1051,640755,2025-01-14,2,23,51.521113,-0.078869,"""Norton Folgate, Liverpool Stre…",51.534042,-0.086379,"""Shoreditch Park, Hoxton"""
2025-01-14 23:59:00,1159,2025-01-15 00:10:00,1007,642548,2025-01-14,2,23,51.522853,-0.099994,"""Berry Street, Clerkenwell""",51.51477,-0.122219,"""Drury Lane, Covent Garden"""
2025-01-14 23:58:00,200048,2025-01-15 00:14:00,1058,944185,2025-01-14,2,23,51.493978,-0.127554,"""Page Street, Westminster""",51.510212,0.004979,"""Orchard Place, Blackwall Tunne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-16 00:02:00,200141,2025-12-16 00:16:00,200237,837578,2025-12-16,2,0,51.509158,-0.224103,"""Westfield Ariel Way, White Cit…",51.472817,-0.199783,"""Parson's Green , Parson's Gree…"
2025-12-16 00:01:00,200011,2025-12-16 00:16:00,200250,908383,2025-12-16,2,0,51.519265,-0.021345,"""Furze Green, Bow""",51.520893,-0.051394,"""Cleveland Way, Stepney"""
2025-12-16 00:00:00,300096,2025-12-16 00:07:00,962,432164,2025-12-16,2,0,51.511891,-0.107349,"""Tallis Street, Temple""",51.505569,-0.111606,"""Stamford Street, South Bank"""


In [45]:
'''
#Load df
#df = pl.read_csv("../data/output/v2/2.clean_df_v2.csv")


df = df.with_columns([
    pl.col("start_date").str.to_datetime(strict=False),
    pl.col("end_date").str.to_datetime(strict=False),
    pl.col("operating_date").str.to_date(strict=False),
])
df
'''

'\n#Load df\n#df = pl.read_csv("../data/output/v2/2.clean_df_v2.csv")\n\n\ndf = df.with_columns([\n    pl.col("start_date").str.to_datetime(strict=False),\n    pl.col("end_date").str.to_datetime(strict=False),\n    pl.col("operating_date").str.to_date(strict=False),\n])\ndf\n'

In [46]:
#########################
#Create station profiles
#########################

#-------------------
#departure features
#-------------------
departures = (
    df
    .group_by("start_station")
    .agg([
        pl.len().alias("n_departures"), #number of trips starting at this station

        pl.col("duration_ms").mean().alias("mean_duration"), #average trip duration for trips departing from that station.
        pl.col("duration_ms").median().alias("median_duration"), #median trip duration departing from that station.
        pl.col("duration_ms").std().alias("std_duration"), #standard deviation trip duration departing from that station.

        pl.col("end_station").n_unique().alias("n_destinations"), #Counts the number of different destination stations reached from each starting station.
    ])
    .rename({"start_station": "station"})
)


#--------------------
#arrival features
#--------------------
arrivals = (
    df
    .group_by("end_station")
    .agg([
        pl.len().alias("n_arrivals"), #number of trips arriving at this station

        pl.col("start_station").n_unique().alias("n_origins"), #Counts the number of different destination stations that end in each end station.
    ])
    .rename({"end_station": "station"})
)


#Join them
stations = departures.join(
    arrivals,
    on="station",
    how="full"
).fill_null(0)

stations

station,n_departures,mean_duration,median_duration,std_duration,n_destinations,station_right,n_arrivals,n_origins
i64,u32,f64,f64,f64,u32,i64,u32,u32
300087,7100,3.7196e6,634420.5,1.5408e8,422,300087,6716,433
1087,11477,1.2755e6,836608.0,1.0489e7,664,1087,11954,664
200053,3280,2.1102e6,925209.5,4.0010e7,466,200053,3097,426
300051,10405,1.3385e6,876639.0,1.2515e7,515,300051,10345,543
300069,11232,1.8484e6,705764.5,8.2752e7,569,300069,10123,537
…,…,…,…,…,…,…,…,…
200249,14178,1.4207e6,663074.0,3.0066e7,535,200249,13595,462
200148,14070,1.3533e6,609900.5,2.3829e7,537,200148,13094,509
200127,4970,1.8336e6,965054.5,9.4181e6,379,200127,5497,420


In [47]:
#-------------------
#departure/arrival ratio
#-------------------

stations = stations.with_columns(
    (pl.col('n_departures')/pl.col('n_arrivals')).alias('departure_arrival_ratio')
)

stations

station,n_departures,mean_duration,median_duration,std_duration,n_destinations,station_right,n_arrivals,n_origins,departure_arrival_ratio
i64,u32,f64,f64,f64,u32,i64,u32,u32,f64
300087,7100,3.7196e6,634420.5,1.5408e8,422,300087,6716,433,1.057177
1087,11477,1.2755e6,836608.0,1.0489e7,664,1087,11954,664,0.960097
200053,3280,2.1102e6,925209.5,4.0010e7,466,200053,3097,426,1.059089
300051,10405,1.3385e6,876639.0,1.2515e7,515,300051,10345,543,1.0058
300069,11232,1.8484e6,705764.5,8.2752e7,569,300069,10123,537,1.109553
…,…,…,…,…,…,…,…,…,…
200249,14178,1.4207e6,663074.0,3.0066e7,535,200249,13595,462,1.042883
200148,14070,1.3533e6,609900.5,2.3829e7,537,200148,13094,509,1.074538
200127,4970,1.8336e6,965054.5,9.4181e6,379,200127,5497,420,0.90413


In [48]:
#-------------------
#Avg departures and arrivals per day
#-------------------
daily_departures = (
    df
    .group_by(["operating_date", "start_station"])
    .agg(
        pl.len().alias("departures")
    )
)

daily_arrivals = (
    df
    .group_by(["operating_date", "end_station"])
    .agg(
        pl.len().alias("arrivals")
    )
)

avg_departures = (
    daily_departures
    .group_by("start_station")
    .agg(
        pl.col("departures").mean().alias("avg_daily_departures")
    )
    .rename({"start_station": "station"})
)

avg_arrivals = (
    daily_arrivals
    .group_by("end_station")
    .agg(
        pl.col("arrivals").mean().alias("avg_daily_arrivals")
    )
    .rename({"end_station": "station"})
)

station_daily_avg = (
    avg_departures
    .join(avg_arrivals, on="station", how="left")
    .sort("station")
)

station_daily_avg

station,avg_daily_departures,avg_daily_arrivals
i64,f64,f64
959,25.78022,26.406593
960,100.381868,121.17033
961,28.717033,28.89011
962,30.870523,32.909091
963,33.346154,38.635616
…,…,…
200191444,9.115385,9.192308
200230444,58.301075,56.308511
300032444,26.478261,25.73913


In [49]:
#-----------------------
#Weekend and weekday behavior
#-----------------------
print(
    df
    .select("day_of_week")
    .unique()
    .sort("day_of_week")
)

df = df.with_columns(
    (pl.col("day_of_week") >= 6).alias("is_weekend")
)

print(df)

weekend_profile = (
    df
    .group_by("start_station")
    .agg([
        pl.col("is_weekend").mean().alias("weekend_ratio")
    ])
    .rename({"start_station": "station"})
)

weekend_profile

shape: (7, 1)
┌─────────────┐
│ day_of_week │
│ ---         │
│ i8          │
╞═════════════╡
│ 1           │
│ 2           │
│ 3           │
│ 4           │
│ 5           │
│ 6           │
│ 7           │
└─────────────┘
shape: (8_762_355, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ start_dat ┆ start_sta ┆ end_date  ┆ end_stati ┆ … ┆ end_stati ┆ end_stati ┆ end_stati ┆ is_weeke │
│ e         ┆ tion      ┆ ---       ┆ on        ┆   ┆ on_latitu ┆ on_longit ┆ on_name   ┆ nd       │
│ ---       ┆ ---       ┆ datetime[ ┆ ---       ┆   ┆ de        ┆ ude       ┆ ---       ┆ ---      │
│ datetime[ ┆ i64       ┆ μs]       ┆ i64       ┆   ┆ ---       ┆ ---       ┆ str       ┆ bool     │
│ μs]       ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025-01-1 ┆ 1043      ┆ 2025-01-1 ┆ 200149    

station,weekend_ratio
i64,f64
3451,0.246941
200202,0.274472
2635,0.189189
22175,0.281195
1227,0.278899
…,…
2587,0.122858
300036,0.238714
300039,0.305184


In [50]:
#--------------------
#Rush hour features
#--------------------

df = df.with_columns([
    (
        (pl.col("hour") >= 7) &
        (pl.col("hour") <= 9)
    ).alias("morning_rush"),

    (
        (pl.col("hour") >= 17) &
        (pl.col("hour") <= 19)
    ).alias("evening_rush"),

    (
        (pl.col("hour") >= 0) &
        (pl.col("hour") <= 5)
    ).alias("night")
])

rush_profile = (
    df
    .group_by("start_station")
    .agg([
        pl.col("morning_rush").mean().alias("morning_rush_ratio"),
        pl.col("evening_rush").mean().alias("evening_rush_ratio"),
        pl.col("night").mean().alias("night_ratio"),
    ])
    .rename({"start_station": "station"})
)

rush_profile

station,morning_rush_ratio,evening_rush_ratio,night_ratio
i64,f64,f64,f64
1182,0.162008,0.216904,0.037155
200065,0.310618,0.153724,0.012678
200062,0.258586,0.168042,0.031891
1054,0.295089,0.213613,0.044119
3436,0.113115,0.289939,0.016673
…,…,…,…
1152,0.076098,0.390544,0.017561
1000,0.242276,0.233979,0.040656
1167,0.319267,0.274001,0.007213


In [51]:
#-------------------
#Final df -> stations
#-------------------

stations = (
    stations
    #.join(hour_profile, on="station", how="left")
    .join(station_daily_avg, on="station", how="left")
    .join(weekend_profile, on="station", how="left")
    .join(rush_profile, on="station", how="left")
    .fill_null(0)
)

print(stations.shape)

(822, 16)


In [52]:
stations = stations.select(['station','n_departures','avg_daily_departures','n_arrivals','avg_daily_arrivals','departure_arrival_ratio','n_destinations','n_origins','mean_duration','median_duration','std_duration','weekend_ratio','morning_rush_ratio','evening_rush_ratio','night_ratio'])

stations

station,n_departures,avg_daily_departures,n_arrivals,avg_daily_arrivals,departure_arrival_ratio,n_destinations,n_origins,mean_duration,median_duration,std_duration,weekend_ratio,morning_rush_ratio,evening_rush_ratio,night_ratio
i64,u32,f64,u32,f64,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64
300087,7100,19.505495,6716,18.450549,1.057177,422,433,3.7196e6,634420.5,1.5408e8,0.241268,0.296056,0.162113,0.044648
1087,11477,31.53022,11954,32.840659,0.960097,664,664,1.2755e6,836608.0,1.0489e7,0.170428,0.255642,0.272197,0.015684
200053,3280,9.111111,3097,8.53168,1.059089,466,426,2.1102e6,925209.5,4.0010e7,0.21372,0.217378,0.283841,0.031707
300051,10405,28.585165,10345,28.42033,1.0058,515,543,1.3385e6,876639.0,1.2515e7,0.241711,0.35925,0.171552,0.017107
300069,11232,30.857143,10123,27.81044,1.109553,569,537,1.8484e6,705764.5,8.2752e7,0.263088,0.319177,0.17557,0.029113
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
200249,14178,38.950549,13595,37.348901,1.042883,535,462,1.4207e6,663074.0,3.0066e7,0.185922,0.114403,0.247143,0.032092
200148,14070,38.653846,13094,35.972527,1.074538,537,509,1.3533e6,609900.5,2.3829e7,0.267448,0.208458,0.201137,0.027079
200127,4970,13.653846,5497,15.101648,0.90413,379,420,1.8336e6,965054.5,9.4181e6,0.364789,0.214085,0.180885,0.030181


In [53]:
#######################
#export df
#######################

stations.write_csv("../data/output/v2/4.stations_df.csv")
